These sql queries were used to pull data from the demo project

1. Events Master
- This is the main raw table. It includes all event types, autocapture detail, URL context, device/browser/geo fields, and person identifiers.

In [ ]:
SELECT
    event,
    timestamp AS event_timestamp,
    distinct_id,

    person.id AS person_id,
    person.properties.email AS person_email,
    if(person.id IS NULL, 0, 1) AS is_identified,

    properties.$session_id AS session_id,
    properties.$current_url AS current_url,
    properties.$pathname AS pathname,
    properties.$host AS host,
    properties.$referrer AS referrer,
    properties.$referring_domain AS referring_domain,

    properties.$event_type AS event_type,
    properties.$el_text AS element_text,
    properties.$el_href AS element_href,
    properties.$elements_chain AS elements_chain,

    properties.$browser AS browser,
    properties.$browser_version AS browser_version,
    properties.$browser_language AS browser_language,
    properties.$os AS os,
    properties.$device_type AS device_type,
    properties.$screen_width AS screen_width,
    properties.$screen_height AS screen_height,
    properties.$viewport_width AS viewport_width,
    properties.$viewport_height AS viewport_height,
    properties.$geoip_country_name AS country,
    properties.$geoip_subdivision_1_name AS region,
    properties.$geoip_city_name AS city,
    properties.$geoip_time_zone AS timezone,

    properties.$lib AS sdk_library,
    properties.$lib_version AS sdk_version,

    properties
FROM events
WHERE properties.$host = 'demo.parsewise.ai'
ORDER BY timestamp ASC

2. Sessions Summary
- One row per session.

In [ ]:
SELECT
    properties.$session_id AS session_id,
    min(timestamp) AS start_timestamp,
    max(timestamp) AS end_timestamp,
    dateDiff('second', min(timestamp), max(timestamp)) AS session_duration_seconds,

    countIf(event = '$pageview') AS pageview_count,
    countIf(event = '$autocapture') AS autocapture_count,

    argMin(properties.$current_url, timestamp) AS entry_url,
    argMin(properties.$pathname, timestamp) AS entry_pathname,
    argMin(properties.$referring_domain, timestamp) AS entry_referring_domain,

    argMax(properties.$current_url, timestamp) AS exit_url,
    argMax(properties.$pathname, timestamp) AS exit_pathname,
    argMax(properties.$pathname, timestamp) AS end_pathname,

    any(distinct_id) AS distinct_id,

    if(countIf(event != '$pageview') = 0 AND countIf(event = '$pageview') <= 1, 1, 0) AS is_bounce
FROM events
WHERE properties.$host = 'demo.parsewise.ai'
  AND properties.$session_id IS NOT NULL
GROUP BY properties.$session_id
ORDER BY start_timestamp ASC

3. Persons
- Detailed view of who they are

In [ ]:
SELECT
    id AS person_id,
    created_at,
    properties.email AS email,
    properties.auth0_org_id AS auth0_org_id,
    properties.initial_referrer AS initial_referrer,
    properties.initial_referring_domain AS initial_referring_domain,
    properties.initial_country AS initial_country,
    properties.initial_device_type AS initial_device_type,
    properties.initial_os AS initial_os,
    properties.initial_browser AS initial_browser,
    properties.initial_channel_type AS initial_channel_type,
    properties.utm_source AS utm_source,
    properties.utm_medium AS utm_medium,
    properties.utm_campaign AS utm_campaign,
    properties.gclid AS gclid,
    properties.fbclid AS fbclid,
    properties.li_fat_id AS li_fat_id
FROM persons
ORDER BY created_at ASC